# 03｜Generate Tables and PNG Figures from Saved Results
Generate LaTeX tables and figures from the actual repetition-level results stored in results/. Figures are first rendered as intermediate vector PDF files and then rasterized into 300 dpi PNG images using Poppler's pdftoppm. The intermediate PDF files are deleted afterward, so only PNG files are retained in figures/.


In [1]:
"""Generate vector figures and LaTeX tables directly from saved raw results.

No simulated or hand-entered measurements appear in this file.
"""
from pathlib import Path
import csv
import json
import math
import numpy as np
from reportlab.pdfgen import canvas
from reportlab.lib.colors import Color, HexColor
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
import reportlab

ROOT = Path.cwd().resolve()
BLUE=HexColor("#176b8a")
ORANGE=HexColor("#c96429")
GREEN=HexColor("#277d59")
GRAY=HexColor("#66707b")
FONTDIR=Path(reportlab.__file__).parent/"fonts"
pdfmetrics.registerFont(TTFont("RoundSans",str(FONTDIR/"Vera.ttf")))
pdfmetrics.registerFont(TTFont("RoundSansBold",str(FONTDIR/"VeraBd.ttf")))


def new_canvas(path):
    return canvas.Canvas(str(path),pagesize=(740,320),initialFontName="RoundSans",
                         initialFontSize=12,invariant=1)


def rows(name):
    with (ROOT/"results"/name).open(encoding="utf-8") as f:
        return list(csv.DictReader(f))


def axes(c,x,y,w,h,title,xlabel,ylabel,xlim,ylim,logx=False,logy=False):
    c.setFont("RoundSansBold",16);c.setFillColorRGB(0,0,0);c.drawString(x,y+h+15,title)
    def tx(v):
        a,b=xlim
        if logx: v,a,b=math.log10(v),math.log10(a),math.log10(b)
        return x+w*(v-a)/(b-a)
    def ty(v):
        a,b=ylim
        if logy: v,a,b=math.log10(v),math.log10(a),math.log10(b)
        return y+h*(v-a)/(b-a)
    c.setStrokeColor(GRAY);c.setLineWidth(.6)
    c.line(x,y,x+w,y);c.line(x,y,x,y+h)
    c.setFont("RoundSans",12)
    xt=np.geomspace(*xlim,4) if logx else np.linspace(*xlim,5)
    yt=np.geomspace(*ylim,5) if logy else np.linspace(*ylim,5)
    for v in xt:
        c.line(tx(v),y,tx(v),y-3)
        c.drawCentredString(tx(v),y-14,f"{v:.3g}")
    for v in yt:
        c.line(x-3,ty(v),x,ty(v))
        c.drawRightString(x-6,ty(v)-3,f"{v:.3g}")
    c.setFont("RoundSans",13);c.drawCentredString(x+w/2,y-35,xlabel)
    #c.saveState();c.translate(x-48,y+h/2);c.rotate(90);c.drawCentredString(0,0,ylabel);c.restoreState()
    c.saveState();c.translate(x-66,y+h/2);c.rotate(90);c.drawCentredString(0,0,ylabel);c.restoreState()
    return tx,ty

def plot_scatter(summary):
    c=new_canvas(ROOT/"figures"/"order_sensitivity.pdf")
    for k,name in enumerate(summary):
        rs=[r for r in rows(f"{name}_order_sensitivity.csv") if int(r["n"])==5000]
        xs=np.array([float(r["dimension"]) for r in rs]);ys=np.array([float(r["query_rmse"]) for r in rs])
        times=np.array([float(r["median_ms"]) for r in rs])
        span=max(float(np.ptp(xs)),1)
        yspan=max(float(np.ptp(ys)),float(ys.mean())*.2,1e-8)
        x0 = 90 + k*370            
        tx,ty=axes(c, x0, 95, 260, 160,   
                   name.upper(),"Parameter dimension","Query RMSE (n = 5000)",
                   (float(xs.min()-.08*span), float(xs.max()+.08*span)),
                   (max(0, float(ys.min()-.15*yspan)), float(ys.max()+1.0*yspan)))
        for x,y,t in zip(xs,ys,times):
            a=(t-times.min())/(times.max()-times.min()+1e-12)
            c.setFillColor(Color(.10+.68*a,.45-.1*a,.58-.4*a))
            c.circle(tx(x),ty(y),3,stroke=0,fill=1)
        c.setFont("RoundSans",11);c.setFillColor(GRAY)
        c.drawString(x0,24,f"100 orders; {summary[name]['unique_graphs']} graphs; points overlap")
        c.drawString(x0,10,f"Time color: {times.min():.3f} to {times.max():.3f} ms")
    c.save()
    

def plot_bounds(summary):
    c=new_canvas(ROOT/"figures"/"bounds.pdf")
    for k,(name,s) in enumerate(summary.items()):
        bs=s["bounds"]
        vals=[r[key] for r in bs for key in ["median_kl","median_oracle_bound","median_empirical_bound"]]
        tx,ty=axes(c, 85+k*350, 75, 250, 185,name.upper(),"Conditional fitting sample size","Median KL / upper bound",
                   (80,6500),(min(vals)*.5,max(vals)*2),True,True)
        for key,color,label in [("median_kl",BLUE,"Actual KL"),("median_oracle_bound",GREEN,"Oracle-weighted"),
                                ("median_empirical_bound",ORANGE,"Empirical confidence bound")]:
            c.setStrokeColor(color);c.setFillColor(color);c.setLineWidth(1.3)
            pts=[(tx(r["n"]),ty(r[key])) for r in bs]
            for (x,y),(xx,yy) in zip(pts,pts[1:]):c.line(x,y,xx,yy)
            for x,y in pts:c.circle(x,y,2.5,stroke=0,fill=1)
        c.setFillColor(GRAY);c.setFont("RoundSans",11)
        #c.drawString(65+k*370,18,"Worst-cell theorem: not applicable.")
        c.drawString(85+k*350,18,"Worst-cell theorem: not applicable.")
    for j,(color,label) in enumerate([(BLUE,"Actual KL"),(GREEN,"Oracle-weighted"),(ORANGE,"Empirical confidence bound")]):
        c.setFillColor(color);c.rect(115+j*200,300,10,3,fill=1,stroke=0)
        c.setFillColorRGB(0,0,0);c.setFont("RoundSans",11);c.drawString(130+j*200,298,label)
    c.save()


def plot_selection(summary):
    methods=["holdout","minimum_dimension","pilot_heuristic","random_order","empirical_bound","induced"]
    labels=["Holdout","Min dim.","Pilot","Random","Emp. bound","Induced"]
    c=new_canvas(ROOT/"figures"/"selection.pdf")
    for k,(name,s) in enumerate(summary.items()):
        rs={r["method"]:r for r in s["selection"] if r["n"]==5000}
        high=max(rs[m]["mean_kl"]+rs[m]["sd_kl"]/math.sqrt(20) for m in methods)*1.2
        #x0,y0,w,h=65+k*370,80,280,180
        x0,y0,w,h=100+k*370,80,260,180
        tx,ty=axes(c,x0,y0,w,h,name.upper(),"","Mean joint KL (n = 5000)",(0,6),(0,high))
        # Cover generic x ticks and provide categorical labels.
        c.setFillColorRGB(1,1,1);c.rect(x0-8,y0-40,w+20,37,stroke=0,fill=1)
        for j,(m,label) in enumerate(zip(methods,labels)):
            x=x0+(j+.5)*w/6
            color=ORANGE if m=="induced" else BLUE
            c.setFillColor(color);v=rs[m]["mean_kl"];se=rs[m]["sd_kl"]/math.sqrt(20)
            c.rect(x-13,y0,26,ty(v)-y0,stroke=0,fill=1)
            c.setStrokeColorRGB(0,0,0);c.setLineWidth(.6);c.line(x,ty(v-se),x,ty(v+se))
            c.line(x-3,ty(v+se),x+3,ty(v+se))
            c.setFont("RoundSans",11);c.setFillColorRGB(0,0,0)
            c.saveState();c.translate(x-5,y0-9);c.rotate(35);c.drawRightString(0,0,label);c.restoreState()
        c.setFont("RoundSans",11);c.setFillColor(GRAY)
        c.drawString(x0,10,"Mean +/- one SE; 20 paired repetitions.")
    c.save()


def make_tables(summary):
    out=[]
    def table(caption,label,cols,head,body):
        chunk=[r"\begin{table}[tbp]\centering",r"\caption{"+caption+r"}\label{"+label+"}",
                    r"\begin{tabular}{"+cols+"}",r"\toprule",head+r"\\",r"\midrule",
                    *[r+r"\\" for r in body],r"\bottomrule\end{tabular}\end{table}"]
        out.extend(chunk)
        (ROOT/"results"/(label.replace("tab:","table_")+".tex")).write_text("\n".join(chunk)+"\n",encoding="utf-8")
    table("New fixed-CPT benchmarks and randomized-order checks. All 100 full and restricted orders were distinct in each network.",
          "tab:new-structure","lrrrrrr",
          "Network & $p$ & $|E|$ & Graphs & $d_{\\min}$ & $d_{\\max}$ & Max. query error",
          [f"{name.upper()} & {s['nodes']} & {s['original_edges']} & {s['unique_graphs']} & {s['dimension_min']} & {s['dimension_max']} & {s['exact_query_max_abs_error']:.2g}" for name,s in summary.items()])
    table("Descriptive constructor timing across 100 orders (milliseconds). Only FGCR is timed; there is no ROS speed claim.",
          "tab:new-time","lrrr","Network & Median & Lower quartile & Upper quartile",
          [f"{name.upper()} & {s['constructor_ms_median']:.4f} & {s['constructor_ms_q25']:.4f} & {s['constructor_ms_q75']:.4f}" for name,s in summary.items()])
    methods=[("holdout","Holdout"),("minimum_dimension","Minimum dimension"),
             ("pilot_heuristic","Pilot heuristic"),("random_order","Random order"),
             ("empirical_bound","Empirical bound"),("induced","Induced subgraph")]
    body=[]
    for key,label in methods:
        row=[label]
        for name,s in summary.items():
            r=next(r for r in s["selection"] if r["method"]==key and r["n"]==5000)
            row.extend([f"{r['mean_kl']:.5f}",f"{r['query_rmse']:.5g}"])
        body.append(" & ".join(row))
    table("Selection at $n=5000$: mean joint KL and scalar-event RMSE over 20 paired repetitions. All methods use the same fitting observations.",
          "tab:new-selection","lrrrr",
          "Selector & CHILD KL & CHILD RMSE & ALARM KL & ALARM RMSE",body)
    body=[]
    for name,s in summary.items():
        for r in s["bounds"]:
            body.append(f"{name.upper()} & {r['n']} & {r['fits']} & {r['median_kl']:.4f} & {r['median_oracle_bound']:.2f} & {r['median_empirical_bound']:.2f}")
    table("Bound behavior at nominal 95\\% per fixed graph. Medians pool distinct conditional graphs and 20 repetitions, not 100 independent graph samples.",
          "tab:new-bounds","lrrrrr","Network & $n$ & Fits & Actual KL & Oracle & Empirical",body)
    (ROOT/"results"/"tables.tex").write_text("\n".join(out)+"\n",encoding="utf-8")


def main():
    summary=json.loads((ROOT/"results"/"summary.json").read_text())
    assert set(summary)=={"child","alarm"},"Both experiments must finish before publishing tables"
    (ROOT/"figures").mkdir(exist_ok=True)
    plot_scatter(summary);plot_bounds(summary);plot_selection(summary);make_tables(summary)
    print("Three figures and tables generated from actual saved outputs.")


In [2]:
main()

import shutil
import subprocess

figure_dir = Path.cwd().resolve() / "figures"
pdftoppm = shutil.which("pdftoppm")
if pdftoppm is None:
    raise FileNotFoundError("未找到 pdftoppm。请安装 Poppler 并将其加入 PATH。")
for pdf_path in sorted(figure_dir.glob("*.pdf")):
    png_prefix = pdf_path.with_suffix("")
    subprocess.run([pdftoppm, "-png", "-singlefile", "-r", "300", str(pdf_path), str(png_prefix)], check=True)
    pdf_path.unlink()
print("PNG 图已写入：", figure_dir)


Three figures and tables generated from actual saved outputs.
PNG 图已写入： D:\jupyternoteboook\实验\新增实验\figures
